In [ ]:
# -*- coding: utf-8 -*-
"""H2B2A7-INTEGRACION-01 CERT (una celda, cero entrenamiento, cero IBM)
Auditoria de las rutas 1E y 1F: identidad del modelo, orden Pauli, U/RXX, diagonal D,
score completo INNER VAL y procedencia de ficheros. No usa OUTER HOLD.
Dependencias: numpy; qiskit requerido para certificar QPY de 1E.
"""
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, csv, os, traceback, sys
import numpy as np

# ================== EDITA SOLO ESTA RUTA SI FUESE NECESARIO ==================
ROOT_OVERRIDE = r'C:\Users\javie\OneDrive\Escritorio\Tesis4'
N_PRUEBAS_RHO = 10  # muestras INNER VAL, para pruebas de matrices y bit ordering
TOL_UNITARIA = 1e-9
TOL_SCORE_REFERENCIA = 2e-5  # tolerancia de AUC archivada, NO reentrenamiento
# ===========================================================================
VERSION = 'H2B2A7-INTEGRACION-01-CERT-v1.0'
FECHA = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ROOT = Path(ROOT_OVERRIDE)
if not (ROOT/'ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz').is_file():
    opts = [Path.cwd(),*Path.cwd().parents,Path.home()/'OneDrive'/'Escritorio'/'Tesis4']
    ROOT = next((p for p in opts if (p/'ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz').is_file()),ROOT)
if not (ROOT/'ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz').is_file():
    raise FileNotFoundError('No se encuentra Tesis4. Corrige ROOT_OVERRIDE, sin mover checkpoints.')
DEST = ROOT/'H2B2A7_PAULI_AUDIT'/('H2B2A7_INTEGRACION_01_CERT_SOLO_LOCAL_'+FECHA)
DEST.mkdir(parents=True,exist_ok=False)
print('='*96)
print('H2B2A7-INTEGRACION-01 CERT | 1E y 1F | SIN ENTRENAMIENTO, OUTER NI IBM')
print('ROOT:',ROOT,'\nSALIDA:',DEST)
print('='*96,flush=True)

def hashfile(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()

def save_json(filename,x):
    (DEST/filename).write_text(json.dumps(x,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

def save_csv(filename,rows,cols=None):
    rows=list(rows);cols=cols or (list(rows[0]) if rows else ['detalle'])
    with open(DEST/filename,'w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore');w.writeheader();w.writerows(rows)

def check(condition,name,detail='',critical=True):
    ok=bool(condition)
    tests.append(dict(prueba=name,estado='PASS' if ok else ('FAIL' if critical else 'RESERVA'),detalle=str(detail)))
    print(('  ✓ ' if ok else '  ✗ ')+name+(f' | {detail}' if detail else ''),flush=True)
    return ok

def read_npz(path,req=()):
    with np.load(path,allow_pickle=False) as f:
        missing=set(req)-set(f.files)
        if missing:raise RuntimeError(f'{path.name}: faltan claves {sorted(missing)}')
        # No ejecutar objetos pickle ni cargar campos no requeridos por la auditoria.
        allowed=set(req)|({'val_auc','final_val_auc'} if 'local_angles' in req else set())
        return {k:np.asarray(f[k]) for k in f.files if k in allowed}

files={
 'TEACHER':ROOT/'ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz',
 'SAFE':ROOT/'ARCH4FH2B2A4_1D2_SAFE_LOCAL_MODEL.npz',
 'OPS_B1B2':ROOT/'ARCH4FH2B2A5_0_OPERATORS_AND_EIGENBASES.npz',
 'X6':ROOT/'ARCH4FH2B1D2_X6_PAULI_EXPECTATIONS.npy',
 'STUDENT_1E':ROOT/'ARCH4FH2B2A5_1E_RANKING_SAFE_MODEL.npz',
 'QPY_1E':ROOT/'ARCH4FH2B2A5_1E_RANKING_SAFE_CIRCUITS.qpy',
 'STUDENT_1F':ROOT/'H2B2A5_1F_LABEL_AWARE'/'best_model_1f.npz',
 'A6_2B1':ROOT/'H2B2A6_QAS'/'DIAGONALES_2B1'/'tabla_final.csv',
 'A6_2B2':ROOT/'H2B2A6_QAS'/'JOINT_2B2'/'tabla_progresiva.csv',
}
# Huellas previamente observadas en INTEGRACION-00: NO inventar SHA del 1F.
known_hash={
 'TEACHER':'0d851421e8ad0e370c3cc365dc053e9ad53b8a5ac5c0d437c72e61df67aa9b89',
 'SAFE':'e203c6e3d36266b20b859bce8f2f4e157dd95e82dde41a656d580d1cdc39b256',
 'X6':'5c76826e8f2e64b3effb943beacbce8cb12f0e4f883afa14372a8f5fdbefaee4',
 'STUDENT_1E':'aa18a34dca6284dd8ad17b9c2565d1132fa58f612fc59356eb9df4aa34d41bfd',
 'QPY_1E':'e6ef44e89f63624f3f0049d5dee45f0ff32e160e7bfb8ddd7d0752dfdd62f9ec',
}
tests=[];inventory=[];scores=[];math_report={};model_data={}
for key,path in files.items():
    row=dict(tipo=key,ruta=str(path),existe=path.is_file(),sha256='',sha_previamente_fijado=known_hash.get(key,''),sha_coincide='NO_PREEXISTENTE')
    if path.is_file():
        row['sha256']=hashfile(path)
        row['sha_coincide']='SI' if key in known_hash and row['sha256']==known_hash[key] else ('NO' if key in known_hash else 'SIN_HUELLA_CONGELADA')
    inventory.append(row)
    if key in known_hash:
        check(row['sha_coincide']=='SI',f'INTEGRIDAD_SHA_{key}',row['sha_coincide'])
    else:
        check(row['existe'],f'EXISTENCIA_{key}', 'SHA calculado sin original de contraste' if row['existe'] else 'Ausente',critical=key in {'OPS_B1B2','STUDENT_1F'})
save_csv('INTEGRACION01_INVENTARIO_Y_HUELLAS.csv',inventory)

# Pruebas algebraicas independientes, separadas de la reconstruccion de score.
X=np.array([[0,1],[1,0]],complex)
Y=np.array([[0,-1j],[1j,0]],complex)
Z=np.diag([1,-1]).astype(complex)
I=np.eye(2,dtype=complex)
SZ={'I':I,'X':X,'Y':Y,'Z':Z}

def pauli_explicit(label):
    P=np.array([[1.]],complex)
    for char in label: P=np.kron(P,SZ[char]) # q5 ... q0
    return P

def p_symplectic(xmask,zmask):
    cols=np.arange(64,dtype=np.int64)
    row=cols^xmask
    out=np.zeros((64,64),complex)
    out[row,cols]=(1j)**((xmask&zmask).bit_count())*np.array([(-1)**((int(c)&zmask).bit_count()) for c in cols])
    return out

def coeff_pauli(O):
    # Convencion original H2B2A6-2A; q0 es el bit menos significativo.
    c=np.arange(64,dtype=np.int64)
    H=np.array([[1-2*((int(col)&z).bit_count()%2) for col in c] for z in range(64)],float)
    perm=(np.arange(64)[:,None]^c[None,:]).astype(int)
    phase=np.array([[(-1j)**((x&z).bit_count()) for z in range(64)] for x in range(64)])
    vals=O[perm,c[None,:]]
    cc=(phase*(vals@H.T)/64.)
    if np.max(np.abs(cc.imag))>2e-8:raise RuntimeError('Error fase de Pauli (parte imaginaria)')
    return cc.real.ravel()

for lab in ['IIIIIY','IIIIIX','IIIZXI','ZYXIII']:
    xx=zz=0
    for q,ch in enumerate(lab[::-1]):
        if ch in ('X','Y'):xx|=1<<q
        if ch in ('Z','Y'):zz|=1<<q
    P=pauli_explicit(lab);Q=p_symplectic(xx,zz)
    check(np.max(np.abs(P-Q))<1e-13,'PAULI_ENDIAN_'+lab,f'error={np.max(np.abs(P-Q)):.1e}')
    cc=coeff_pauli(P)
    idx=xx*64+zz
    check(abs(cc[idx]-1)<1e-12 and np.max(np.abs(np.delete(cc,idx)))<1e-12,'PAULI_COEF_'+lab)

# Topologia EXACTA archivada para 1E y 1F: no aplicar a candidatos A6 modulares.
MATCHINGS=[[(0,5),(1,4),(2,3)],[(0,4),(5,3),(1,2)],
           [(0,3),(4,2),(5,1)],[(0,2),(3,1),(4,5)],
           [(0,1),(2,5),(3,4)]]
ROWS=np.arange(64,dtype=int)
BIT=np.array([(ROWS>>q)&1 for q in range(6)])
FLIPS=np.array([ROWS^(1<<q) for q in range(6)])

def unitary_independent(local,angles):
    """Reconstructor independiente de Qiskit: RZ, RY, RXX en orden historico."""
    if local.shape!=(17,6,2) or angles.shape!=(48,):
        raise ValueError(f'Formas no esperadas {local.shape},{angles.shape}')
    U=np.eye(64,dtype=complex)
    def layer(U,A):
        for q in range(6):
            rz,ry=map(float,A[q]);phase=np.exp(-.5j*rz*(1-2*BIT[q]))
            U=phase[:,None]*U
            U=np.cos(ry/2)*U+np.sin(ry/2)*(2*BIT[q]-1)[:,None]*U[FLIPS[q],:]
        return U
    U=layer(U,local[0]);t=0
    for round_idx in range(16):
        for q,p in MATCHINGS[round_idx%5]:
            theta=float(angles[t]);t+=1
            U=np.cos(theta/2)*U-1j*np.sin(theta/2)*U[ROWS^(1<<q)^(1<<p),:]
        U=layer(U,local[round_idx+1])
    assert t==48
    return U

def complex_cmp_global(A,B):
    # Distancia de matrices hasta fase global; preferible a comparar solo conteos de puertas.
    dot=np.vdot(A.ravel(),B.ravel())
    phase=np.conj(dot)/abs(dot) if abs(dot)>1e-14 else 1
    return float(np.max(np.abs(A-phase*B)))

try:
    import qiskit
    from qiskit import QuantumCircuit, qpy
    from qiskit.quantum_info import Operator, Statevector
    QISKIT_READY=True
    print('Qiskit detectado:',qiskit.__version__)
except Exception as exc:
    QISKIT_READY=False
    check(False,'QISKIT_PARA_QPY',str(exc),critical=True)

# Carga de teacher, rank-2 y OUTER HOLD SOLO para certificar separacion; nunca evaluar OUTER.
try:
    req=['TEACHER','SAFE','OPS_B1B2','X6']
    if not all(files[k].is_file() for k in req):raise FileNotFoundError('Falta alguna fuente principal')
    teach=read_npz(files['TEACHER'],['y','lambdas','baseline','inner_train_idx','inner_val_idx','outer_hold_idx'])
    safe=read_npz(files['SAFE'],['B','C','d','T_train','T_val','znew_train','znew_val'])
    ops=read_npz(files['OPS_B1B2'],['B1_matrix','B2_matrix'])
    X6=np.load(files['X6'],mmap_mode='r',allow_pickle=False)
    tr=np.asarray(teach['inner_train_idx'],int).ravel()
    va=np.asarray(teach['inner_val_idx'],int).ravel()
    hold=np.asarray(teach['outer_hold_idx'],int).ravel()
    n=len(teach['y'])
    check(not(set(tr)&set(va) or set(tr)&set(hold) or set(va)&set(hold)), 'SPLITS_DISJUNTOS',f'TRAIN={len(tr)} VAL={len(va)} HOLD={len(hold)}')
    check(max(np.r_[tr,va,hold])<n and X6.shape==(n,4096),'INDICES_Y_X6',f'X6={X6.shape}')
    L=np.asarray(teach['lambdas'],float);baseline=np.asarray(teach['baseline'],float).ravel()
    T=np.asarray(safe['T_val'],float);z=np.asarray(safe['znew_val'],float)
    C=np.asarray(safe['C'],float);d=np.asarray(safe['d'],float).ravel()
    check(T.shape==(len(va),L.shape[1]) and z.shape==(len(va),2) and C.shape==(2,L.shape[1]),'FORMA_SCORE_SAFE')
    T4=T-z@C-d
    y=np.asarray(teach['y']).ravel()
    if set(np.unique(y))=={-1,1}:y=(y==1).astype(int)
    yva=y[va]
    check(set(np.unique(yva))=={0,1},'ETIQUETAS_VAL_BINARIAS')
    targets=[np.asarray(ops['B1_matrix'],complex),np.asarray(ops['B2_matrix'],complex)]
    for j,B in enumerate(targets):
        check(B.shape==(64,64) and np.max(np.abs(B-B.conj().T))<1e-9,f'HERMITICIDAD_B{j+1}')
        coeff=coeff_pauli(B)
        from_safe=np.asarray(safe['B'],float)
        check(from_safe.shape==(4095,2) and max(abs(coeff[0]),np.max(np.abs(coeff[1:]-from_safe[:,j])))<2e-6,f'OBSERVABLE_B{j+1}_PAULI',f'error={np.max(np.abs(coeff[1:]-from_safe[:,j])):.3e}')
        quick=np.asarray(X6[va[:min(16,len(va))]],float)@coeff
        check(np.max(np.abs(quick-z[:len(quick),j]))<3e-4,f'CACHE_X6_VS_SAFE_B{j+1}',f'max={np.max(np.abs(quick-z[:len(quick),j])):.3e}')
    # Prueba de estado reducido: invertir algebraicamente expectativas X6 -> rho6.
    cidx=np.arange(64,dtype=int)
    H=np.asarray([[1-2*((int(c)&zmask).bit_count()%2) for c in cidx] for zmask in range(64)],float)
    phase=np.asarray([[(1j)**((xmask&zmask).bit_count()) for zmask in range(64)] for xmask in range(64)],complex)
    rindex=cidx[None,:]^cidx[:,None]
    rchecks=[]
    for k in range(min(N_PRUEBAS_RHO,len(va))):
        global_id=int(va[k]);exp=np.asarray(X6[global_id],float).reshape(64,64)
        entries=((exp*phase)@H.T)/64
        rho=np.empty((64,64),complex);rho[rindex,cidx[None,:]]=entries
        herm=float(np.max(np.abs(rho-rho.conj().T)))
        eig=np.linalg.eigvalsh((rho+rho.conj().T)/2)
        trerr=abs(np.trace(rho)-1)
        tail=float(np.max(abs(eig[:-2])))
        berr=max(abs(np.trace(rho@targets[j])-z[k,j]) for j in range(2))
        ok=herm<3e-4 and trerr<5e-4 and tail<3e-4 and min(eig)>-3e-4 and berr<3e-4
        check(ok,'RHO6_RANGO_POSITIVIDAD_'+str(k),f'id={global_id}, herm={herm:.1e}, tr={trerr:.1e}, rango={tail:.1e}, B={berr:.1e}')
        rchecks.append(rho)
    READY_SCORE=True
except Exception as e:
    READY_SCORE=False
    check(False,'DATOS_Y_OBSERVABLE_SAFE',repr(e))


def auc_ranks(labels,values):
    """AUC de Mann-Whitney, con rangos promediados para empates, sin sklearn."""
    labels=np.asarray(labels,int);values=np.asarray(values,float)
    order=np.argsort(values,kind='mergesort');s=values[order]
    rank=np.empty(len(s),float);a=0
    while a<len(s):
        b=a+1
        while b<len(s) and s[b]==s[a]:b+=1
        rank[a:b]=(a+b+1)/2;a=b
    unpack=np.empty(len(s),float);unpack[order]=rank
    n1=int(np.sum(labels==1));n0=int(np.sum(labels==0))
    return float((np.sum(unpack[labels==1])-n1*(n1+1)/2)/(n1*n0))


def qiskit_builder(st,idx):
    qc=QuantumCircuit(6,name=f'H2B2A5_C{idx}')
    a=np.asarray(st['local_angles'][idx],float)
    t=np.asarray(st['rxx_angles'][idx],float)
    for q in range(6):qc.rz(float(a[0,q,0]),q);qc.ry(float(a[0,q,1]),q)
    k=0
    for r in range(16):
        for q,p in MATCHINGS[r%5]:qc.rxx(float(t[k]),q,p);k+=1
        for q in range(6):qc.rz(float(a[r+1,q,0]),q);qc.ry(float(a[r+1,q,1]),q)
    return qc


def audit_student(key,existing_qpy=None):
    path=files[key]
    if not path.is_file():return
    try:
        st=read_npz(path,['local_angles','rxx_angles','diagonal'])
        a=np.asarray(st['local_angles'],float);t=np.asarray(st['rxx_angles'],float);d0=np.asarray(st['diagonal'],float)
        check(a.shape==(2,17,6,2) and t.shape==(2,48) and d0.shape==(2,64),key+'_SHAPES',f'{a.shape},{t.shape},{d0.shape}')
        check(all(np.isfinite(v).all() for v in [a,t,d0]),key+'_FINITE')
        for j in range(2):
            U=unitary_independent(a[j],t[j]);err=float(np.max(np.abs(U.conj().T@U-np.eye(64))))
            check(err<TOL_UNITARIA,key+f'_UNITARIO_{j+1}',f'{err:.3e}')
            O=U.conj().T@(d0[j][:,None]*U)
            check(np.max(np.abs(O-O.conj().T))<TOL_UNITARIA,key+f'_HERMITICO_{j+1}')
            # Prueba independiente de D: suma de probabilidades en base computacional.
            for state_index in [0,1,21,63]:
                v=U[:,state_index];p=np.abs(v)**2
                a1=float(np.vdot(U[:,state_index],d0[j]*U[:,state_index]).real)
                a2=float(np.real(O[state_index,state_index]))
                check(abs(p.sum()-1)<1e-10 and abs(a1-a2)<1e-10,key+f'_DIAG_ENDIAN_B{j+1}_X{state_index}',f'{a1-a2:.2e}')
            if QISKIT_READY:
                qc=qiskit_builder(st,j);Uq=Operator(qc).data
                e=complex_cmp_global(U,Uq)
                check(e<1e-8,key+f'_QISKIT_INDEPENDIENTE_B{j+1}',f'{e:.3e}')
            else:
                check(False,key+f'_QISKIT_INDEPENDIENTE_B{j+1}','qiskit no disponible')
            math_report[key+f'_B{j+1}']={
                'unitaria_error_max':err,
                'norma_diagonal_inf':float(np.max(np.abs(d0[j]))),
                'error_frobenius_vs_Bstar':float(np.linalg.norm(O-targets[j])/(np.linalg.norm(targets[j])+1e-15)) if READY_SCORE else None,
                'diagonal_min':float(np.min(d0[j])), 'diagonal_max':float(np.max(d0[j]))
            }
        if QISKIT_READY and existing_qpy is not None and existing_qpy.is_file():
            with open(existing_qpy,'rb') as handle: circs=qpy.load(handle)
            check(len(circs)==2,key+'_QPY_DOS_CIRCUITOS',f'circuitos={len(circs)}')
            for j,circ in enumerate(circs[:2]):
                Uq=Operator(circ).data; U=unitary_independent(a[j],t[j]);e=complex_cmp_global(U,Uq)
                check(circ.num_qubits==6 and e<1e-8,key+f'_QPY_ARCHIVADO_B{j+1}',f'error={e:.3e}, puertas={Counter(inst.operation.name for inst in circ.data)}')
        if READY_SCORE:
            Ocomp=[unitary_independent(a[j],t[j]).conj().T@(d0[j][:,None]*unitary_independent(a[j],t[j])) for j in range(2)]
            co=np.array([coeff_pauli(Ocomp[j]) for j in range(2)])
            zcalc=np.empty((len(va),2),float)
            for i in range(0,len(va),256):zcalc[i:i+256]=np.asarray(X6[va[i:i+256]],float)@co.T
            for k,rho in enumerate(rchecks):
                delta=max(abs(float(np.trace(rho@Ocomp[j]).real)-zcalc[k,j]) for j in range(2))
                check(delta<3e-4,key+'_EXPECTACION_RHO_VS_X6_'+str(k),f'error={delta:.2e}')
            pred=baseline[va]+np.einsum('ij,ij->i',L[va],T4+zcalc@C+d)
            AUC=auc_ranks(yva,pred)
            stored = st.get('val_auc',st.get('final_val_auc',np.array([])))
            stored=float(np.ravel(stored)[0]) if np.asarray(stored).size else None
            accepted=stored is not None and abs(AUC-stored)<TOL_SCORE_REFERENCIA
            check(accepted,key+'_AUC_INNER_VAL_ARCHIVO',f'AUC calculada={AUC:.9f} | AUC archivo={stored}')
            scores.extend(dict(modelo=key,global_index=int(ind),score=float(sc),split='INNER_VAL_YA_EXAMINADO') for ind,sc in zip(va,pred))
            math_report[key+'_INNER_VAL']={'n':len(va),'auc_recalculada':AUC,'auc_archivada':stored,'error_auc':None if stored is None else abs(AUC-stored),'score_por_material_archivado':False,'nota':'AUC archivada coincide si supera prueba; NO hay vector pareado archivado para equivalencia de cada score.'}
    except Exception as e:
        check(False,key+'_EJECUCION',repr(e))
        (DEST/(key+'_ERROR.txt')).write_text(traceback.format_exc(),encoding='utf-8')

# En 1E se certifica su QPY; 1F no tiene QPY original necesariamente.
audit_student('STUDENT_1E',files['QPY_1E'])
audit_student('STUDENT_1F',None)

save_csv('INTEGRACION01_PRUEBAS_DE_CERTIFICACION.csv',tests)
save_csv('INTEGRACION01_SCORES_INNER_VAL_RECONSTRUIDOS.csv',scores,['modelo','global_index','score','split'])
save_json('INTEGRACION01_DETALLES_MATRICIALES.json',math_report)
failed=[r for r in tests if r['estado']=='FAIL']
reserved=[r for r in tests if r['estado']=='RESERVA']
summary={'experimento':VERSION,'modo':'SOLO_LOCAL_CERTIFICACION_SIN_REENTRENAR','root':str(ROOT),
         'salida':str(DEST),'outer_hold_NO_evaluado':True,'IBM_jobs':0,'QPU_shots':0,
         'QPY_1E_cotejado':any(r['prueba']=='STUDENT_1E_QPY_ARCHIVADO_B1' and r['estado']=='PASS' for r in tests),
         '1E_score_auc_cotejado':any(r['prueba']=='STUDENT_1E_AUC_INNER_VAL_ARCHIVO' and r['estado']=='PASS' for r in tests),
         '1F_score_auc_cotejado':any(r['prueba']=='STUDENT_1F_AUC_INNER_VAL_ARCHIVO' and r['estado']=='PASS' for r in tests),
         'fallos':[r['prueba']+' — '+r['detalle'] for r in failed],
         'reservas':[r['prueba']+' — '+r['detalle'] for r in reserved],
         'estado':'PRUEBAS_NUMERICAS_SUPERADAS_VALIDACION_HISTORICA_PENDIENTE' if not failed and not reserved else 'BLOQUEADO_PENDIENTE_REVISION',
         'limites':['1F y H2B2A6 carecen de SHA anterior comprobable en esta auditoria; huellas nuevas no certifican identidad historica',
                   'AUC coincidente no certifica igualdad por material si no existe score pareado original',
                   'Costo por material y layout ISA siguen pendientes','INNER_VAL ya examinado; no seleccion ni prueba externa']}
save_json('INTEGRACION01_RESUMEN_DECISION.json',summary)
md=['# H2B2A7-INTEGRACION-01 — Certificacion sin entrenamiento','',
    f"**Estado:** {summary['estado']}",'',f'**Directorio:** `{DEST}`','',
    f"**Pruebas:** {len(tests)}; fallos: {len(failed)}; reservas: {len(reserved)}",'',
    '## Alcance','Verifica 1E/1F existentes sobre INNER VAL; no evalua OUTER HOLD ni optimiza parametros.',
    'Reconstruye matrices U de modo independiente, comprueba diagonales D y AUC archivada.',
    'Certifica QPY de 1E unicamente si Qiskit esta instalado; 1F no usa por defecto QPY de 1E.',
    '', '## Fallos o bloqueos',]
md+=['- '+s for s in summary['fallos']] or ['- Ninguno']
md+=['','## Limites y pasos siguientes']+['- '+s for s in summary['limites']]
md+=['','Sin QPU; no se declara equivalencia completa ni ventaja de student por haber aprobado solo esta auditoria.']
(DEST/'INTEGRACION01_INFORME.md').write_text('\n'.join(md)+'\n',encoding='utf-8')
print('\n'+'='*96)
print('ESTADO:',summary['estado'],'| PRUEBAS:',len(tests),'| FALLOS:',len(failed),'| RESERVAS:',len(reserved))
print('STUDENT 1E / 1F, AUC cotejadas:',summary['1E_score_auc_cotejado'],summary['1F_score_auc_cotejado'])
print('QPY 1E verificado:',summary['QPY_1E_cotejado'])
print('RESUMEN:',DEST/'INTEGRACION01_RESUMEN_DECISION.json')
print('INFORME:',DEST/'INTEGRACION01_INFORME.md')
print('NO ENTRENAMIENTO | NO OUTER HOLD | NO IBM | NO SHOTS')
print('='*96)
